# Residual Networks (ResNet) --- PyTorch Port

This is a PyTorch port of `ResNet-Implementation.ipynb` (the original Keras/TensorFlow
from-scratch ResNet-50). It ports the **architecture only**: `identity_block`,
`convolutional_block`, and the `ResNet50` builder, plus the classification head. The
original notebook's dataset download, augmentation, training loop, and evaluation cells
(built around an unrelated Indian Bird Species dataset) are intentionally not ported --
see `Porting notes` at the end of this notebook for every TensorFlow -> PyTorch decision
made along the way.

__ResNet__ or __Residual Neural Network__ was proposed by ([He et al.](https://arxiv.org/pdf/1512.03385.pdf)) researchers at Microsoft Research namely, _Kaiming He_, _Xiangyu Zhang_, _Shaoqing Ren_ and, _Jian Sun_; which allow us to train much deeper networks than were previously practically feasible. Also, ResNet won [ImageNet](https://www.image-net.org/about.php) Challenge in 2015.

## Problem with Very Deep Neural Networks
The main benefit of a very deep network is that it can represent very complex functions. It can also learn features at many different levels of abstraction, from edges (at the lower layers) to very complex features (at the deeper layers). However, using a deeper network doesn't always help. A huge barrier to training them is vanishing gradients: very deep networks often have a gradient signal that goes to zero quickly, thus making gradient descent unbearably slow. More specifically, during gradient descent, as you backprop from the final layer back to the first layer, you are multiplying by the weight matrix on each step, and thus the gradient can decrease exponentially quickly to zero (or, in rare cases, grow exponentially quickly and "explode" to take very large values).

During training, we might therefore see the magnitude (or norm) of the gradient for the earlier layers decrease to zero very rapidly as training proceeds.

## Building a Residual Network
In ResNet architecture, a *"shortcut"* or a *"skip connection"* allows the gradient to be directly backpropagated to earlier layers. By stacking these ResNet blocks on top of each other, you can form a very deep network.

Having ResNet blocks with the shortcut also makes it very easy for one of the blocks to learn an identity function. This means that you can stack on additional ResNet blocks with little risk of harming training set performance.

Two main types of blocks are used in a ResNet, depending mainly on whether the input/output dimensions are same or different. Both are implemented below.

# ResNet-50 Implementation (PyTorch)

## Importing Dependencies

In [ ]:
from __future__ import annotations

import torch
import torch.nn as nn
import torch.nn.functional as F

## Defining the Structure of ResNet50

### Identity Block
The identity block is the standard block used in ResNets, and corresponds to the case where the input activation has the same dimension as the output activation. The upper path is the "shortcut path." The lower path is the "main path," with three CONV2D -> BatchNorm -> ReLU steps, where the skip connection "skips over" 3 hidden layers.

First component of main path:
- CONV2D with $F_1$ filters of shape (1, 1), stride (1, 1), padding "valid" -> BatchNorm -> ReLU.

Second component of main path:
- CONV2D with $F_2$ filters of shape $(f, f)$, stride (1, 1), padding "same" -> BatchNorm -> ReLU.

Third component of main path:
- CONV2D with $F_3$ filters of shape (1, 1), stride (1, 1), padding "valid" -> BatchNorm. No ReLU here.

Final step:
- The shortcut and the main path are added together, then ReLU is applied.

In [ ]:
class IdentityBlock(nn.Module):
    """ResNet identity block: 3-layer skip connection with no dimension change.

    Direct port of the notebook's `identity_block(X, f, filters, stage, block)`.
    `stage`/`block`/named layers existed in Keras purely for `model.summary()`
    readability; PyTorch modules are already addressable by attribute path, so
    those two arguments are dropped here (see "Porting notes").

    Args:
        in_channels: Number of channels entering the block.
        f: Kernel size of the middle (F2) convolution.
        filters: (F1, F2, F3) output-channel counts for the three convolutions.
    """

    def __init__(self, in_channels: int, f: int, filters: tuple[int, int, int]) -> None:
        super().__init__()
        f1, f2, f3 = filters

        self.conv1 = nn.Conv2d(in_channels, f1, kernel_size=1, stride=1, padding=0)
        self.bn1 = nn.BatchNorm2d(f1)

        self.conv2 = nn.Conv2d(f1, f2, kernel_size=f, stride=1, padding=f // 2)
        self.bn2 = nn.BatchNorm2d(f2)

        self.conv3 = nn.Conv2d(f2, f3, kernel_size=1, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(f3)

        self.relu = nn.ReLU(inplace=True)
        for conv in (self.conv1, self.conv2, self.conv3):
            nn.init.xavier_uniform_(conv.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out = out + shortcut
        return self.relu(out)

### Convolution Block
The ResNet "convolutional block" is used when the input and output dimensions don't match. The difference from the identity block is a CONV2D + BatchNorm in the shortcut path (no activation), used to resize the input so dimensions line up for the final addition.

First component of main path:
- CONV2D, $F_1$ filters, (1, 1), stride $(s, s)$, padding "valid" -> BatchNorm -> ReLU.

Second component of main path:
- CONV2D, $F_2$ filters, $(f, f)$, stride (1, 1), padding "same" -> BatchNorm -> ReLU.

Third component of main path:
- CONV2D, $F_3$ filters, (1, 1), stride (1, 1), padding "valid" -> BatchNorm. No ReLU here.

Shortcut path:
- CONV2D, $F_3$ filters, (1, 1), stride $(s, s)$, padding "valid" -> BatchNorm.

Final step:
- The shortcut and the main path are added together, then ReLU is applied.

In [ ]:
class ConvolutionalBlock(nn.Module):
    """ResNet convolutional block: skip connection with a projection shortcut.

    Direct port of the notebook's `convolutional_block(X, f, filters, stage, block, s)`.

    Args:
        in_channels: Number of channels entering the block.
        f: Kernel size of the middle (F2) convolution.
        filters: (F1, F2, F3) output-channel counts for the three convolutions.
        s: Stride applied by the first main-path convolution and the shortcut
            projection, so both paths shrink the spatial dimensions identically.
    """

    def __init__(
        self, in_channels: int, f: int, filters: tuple[int, int, int], s: int = 2
    ) -> None:
        super().__init__()
        f1, f2, f3 = filters

        self.conv1 = nn.Conv2d(in_channels, f1, kernel_size=1, stride=s, padding=0)
        self.bn1 = nn.BatchNorm2d(f1)

        self.conv2 = nn.Conv2d(f1, f2, kernel_size=f, stride=1, padding=f // 2)
        self.bn2 = nn.BatchNorm2d(f2)

        self.conv3 = nn.Conv2d(f2, f3, kernel_size=1, stride=1, padding=0)
        self.bn3 = nn.BatchNorm2d(f3)

        self.shortcut_conv = nn.Conv2d(in_channels, f3, kernel_size=1, stride=s, padding=0)
        self.shortcut_bn = nn.BatchNorm2d(f3)

        self.relu = nn.ReLU(inplace=True)
        for conv in (self.conv1, self.conv2, self.conv3, self.shortcut_conv):
            nn.init.xavier_uniform_(conv.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        shortcut = self.shortcut_bn(self.shortcut_conv(x))

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.relu(self.bn2(self.conv2(out)))
        out = self.bn3(self.conv3(out))

        out = out + shortcut
        return self.relu(out)

## Building ResNet Model (50 Layers)
The ResNet-50 model consists of 5 stages, each with a convolutional block and one or more identity blocks. Each block has 3 convolution layers, for ~23 million trainable parameters total.

- Zero-padding pads the input with a pad of (3, 3).
- Stage 1: 2D Convolution, 64 filters of shape (7, 7), stride (2, 2) -> BatchNorm -> ReLU -> MaxPooling, (3, 3) window, stride (2, 2).
- Stage 2: convolutional block with filters [64, 64, 256], f=3, s=1, then 2 identity blocks with the same filters.
- Stage 3: convolutional block with filters [128, 128, 512], f=3, s=2, then 3 identity blocks with the same filters.
- Stage 4: convolutional block with filters [256, 256, 1024], f=3, s=2, then 5 identity blocks with the same filters.
- Stage 5: convolutional block with filters [512, 512, 2048], f=3, s=2, then 2 identity blocks with the same filters.
- 2D Average Pooling, window (2, 2), padding "same".

In [ ]:
def _tf_same_avg_pool(x: torch.Tensor, kernel_size: int = 2, stride: int = 2) -> torch.Tensor:
    """Replicate Keras/TensorFlow `padding="same"` for AveragePooling2D.

    PyTorch's `nn.AvgPool2d(padding=...)` only pads symmetrically, but TF's "same"
    padding is asymmetric whenever `(output - 1) * stride + kernel_size - input` is
    odd -- exactly the case for this network's 7x7 -> 4x4 pool. Pad explicitly, then
    pool with padding=0, to match TF's output shape exactly.
    """

    def _pad_amount(size: int) -> tuple[int, int]:
        output = -(-size // stride)  # ceil division
        total = max((output - 1) * stride + kernel_size - size, 0)
        before = total // 2
        return before, total - before

    pad_top, pad_bottom = _pad_amount(x.shape[2])
    pad_left, pad_right = _pad_amount(x.shape[3])
    return F.pad(x, (pad_left, pad_right, pad_top, pad_bottom))


class ResNet50(nn.Module):
    """ResNet-50 backbone (no classification head).

    CONV -> BN -> RELU -> MAXPOOL -> CONVBLOCK -> IDBLOCK*2 -> CONVBLOCK -> IDBLOCK*3
    -> CONVBLOCK -> IDBLOCK*5 -> CONVBLOCK -> IDBLOCK*2 -> AVGPOOL

    Direct port of the notebook's `ResNet50(input_shape)` builder. Returns the pooled
    feature map, matching `base_model.output` in the original -- flattening and the
    dense head are a separate module (`ResNet50Classifier` below), same as the
    original notebook builds them in a separate cell.
    """

    def __init__(self, in_channels: int = 3) -> None:
        super().__init__()

        self.zero_pad = nn.ZeroPad2d(3)
        self.conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=0)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.max_pool = nn.MaxPool2d(kernel_size=3, stride=2, padding=0)
        nn.init.xavier_uniform_(self.conv1.weight)

        self.stage2 = nn.Sequential(
            ConvolutionalBlock(64, f=3, filters=(64, 64, 256), s=1),
            IdentityBlock(256, f=3, filters=(64, 64, 256)),
            IdentityBlock(256, f=3, filters=(64, 64, 256)),
        )
        self.stage3 = nn.Sequential(
            ConvolutionalBlock(256, f=3, filters=(128, 128, 512), s=2),
            IdentityBlock(512, f=3, filters=(128, 128, 512)),
            IdentityBlock(512, f=3, filters=(128, 128, 512)),
            IdentityBlock(512, f=3, filters=(128, 128, 512)),
        )
        self.stage4 = nn.Sequential(
            ConvolutionalBlock(512, f=3, filters=(256, 256, 1024), s=2),
            IdentityBlock(1024, f=3, filters=(256, 256, 1024)),
            IdentityBlock(1024, f=3, filters=(256, 256, 1024)),
            IdentityBlock(1024, f=3, filters=(256, 256, 1024)),
            IdentityBlock(1024, f=3, filters=(256, 256, 1024)),
            IdentityBlock(1024, f=3, filters=(256, 256, 1024)),
        )
        self.stage5 = nn.Sequential(
            ConvolutionalBlock(1024, f=3, filters=(512, 512, 2048), s=2),
            IdentityBlock(2048, f=3, filters=(512, 512, 2048)),
            IdentityBlock(2048, f=3, filters=(512, 512, 2048)),
        )

        self.avg_pool = nn.AvgPool2d(kernel_size=2, stride=2, padding=0)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.zero_pad(x)
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.max_pool(x)

        x = self.stage2(x)
        x = self.stage3(x)
        x = self.stage4(x)
        x = self.stage5(x)

        return self.avg_pool(_tf_same_avg_pool(x))

***Defining Base Model***

In [ ]:
base_model = ResNet50(in_channels=3)

***Creating Output Layer***

The head mirrors the original: `Flatten -> Dense(256, relu) -> Dense(128, relu) -> Dense(num_classes)`.
Softmax is intentionally excluded from the final layer -- this repo's own
`CatDogCNN` (`src/cats_dogs_mlops/model.py`) uses the same convention: raw logits
out, `nn.CrossEntropyLoss` applies softmax internally in a numerically stable form.
See "Porting notes" for why this is preferred over literally adding `nn.Softmax`.

In [ ]:
class ResNet50Classifier(nn.Module):
    """ResNet-50 backbone + fully-connected classification head.

    For the default 224x224 input, the backbone's average pool produces a
    (4, 4, 2048) feature map (see `_tf_same_avg_pool` above for the size math:
    224 -> 112 -> 55 -> 28 -> 14 -> 7 -> 4), flattened to 32768 features.

    Args:
        num_classes: Number of output classes for the final Dense layer.
        in_channels: Number of input image channels.
    """

    _FLATTENED_FEATURES = 4 * 4 * 2048

    def __init__(self, num_classes: int, in_channels: int = 3) -> None:
        super().__init__()
        self.backbone = ResNet50(in_channels=in_channels)

        fc1 = nn.Linear(self._FLATTENED_FEATURES, 256)
        fc2 = nn.Linear(256, 128)
        fc3 = nn.Linear(128, num_classes)
        for layer in (fc1, fc2, fc3):
            nn.init.xavier_uniform_(layer.weight)

        self.classifier = nn.Sequential(
            nn.Flatten(),
            fc1,
            nn.ReLU(inplace=True),
            fc2,
            nn.ReLU(inplace=True),
            fc3,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        features = self.backbone(x)
        return self.classifier(features)

In [ ]:
NUMBER_OF_CLASSES = 2  # placeholder; set to your dataset's class count

model = ResNet50Classifier(num_classes=NUMBER_OF_CLASSES)

trainable_parameters = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total trainable parameters: {trainable_parameters:,}")

### `compile` method equivalent

PyTorch has no single `model.compile(...)` call -- optimizer, loss, and metrics are
held as separate objects and used explicitly inside the training loop.

In [ ]:
optimizer = torch.optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()  # softmax + categorical cross-entropy, combined and stable

# accuracy is not a standalone object in PyTorch -- compute it per batch, e.g.:
#   accuracy = (logits.argmax(dim=1) == labels).float().mean()

## Self-check

Non-trivial logic (the block classes, the TF `same`-padding emulation) needs a runnable
check. This forwards a dummy batch through the full model and asserts the output shape,
the same way the original notebook's `model.summary()` cell confirms the architecture
wires together correctly before any real training data is involved.

In [ ]:
def _self_check() -> None:
    dummy_input = torch.randn(2, 3, 224, 224)
    output = model(dummy_input)
    assert output.shape == (2, NUMBER_OF_CLASSES), output.shape

    backbone_output = base_model(dummy_input)
    assert backbone_output.shape == (2, 2048, 4, 4), backbone_output.shape

    print("ResNet50Classifier output shape:", tuple(output.shape))
    print("ResNet50 backbone output shape:", tuple(backbone_output.shape))
    print("Self-check passed.")


_self_check()

## Porting notes (TensorFlow/Keras -> PyTorch)

- **Data layout:** Keras uses channels-last (NHWC); PyTorch uses channels-first (NCHW).
  All conv/BN layers are defined accordingly -- no transpose needed since every tensor
  in this notebook is already NCHW from the start.
- **`kernel_initializer=glorot_uniform(seed=0)`:** Glorot uniform == Xavier uniform
  (same initialization, different name). Applied via `nn.init.xavier_uniform_` on every
  Conv2d/Linear weight, mirroring the original. The `seed=0` argument has no direct
  PyTorch per-layer equivalent; call `torch.manual_seed(0)` once before construction if
  exact reproducibility across runs matters.
- **`padding="same"` / `"valid"`:** `"valid"` is PyTorch's default `padding=0`. `"same"`
  with stride 1 and odd kernel size `f` is `padding=f // 2` (used throughout for the
  `(f, f)` middle convolutions). The one `"same"`-padded pool with stride 2
  (`AveragePooling2D` at the end of the backbone) needed the `_tf_same_avg_pool` helper
  above, because PyTorch's built-in padding is symmetric and this case isn't.
- **Named layers (`stage`, `block`, `conv_name_base`, `bn_name_base`):** existed so
  Keras's `model.summary()` and TensorBoard graphs were human-readable. PyTorch modules
  are already addressable via `model.backbone.stage3[0].conv2`, so those arguments were
  dropped rather than carried over as unused string bookkeeping.
- **Bias:** Keras `Conv2D` defaults to `use_bias=True` and the original never overrides
  it, even though every conv here is immediately followed by BatchNorm (which makes the
  bias redundant). Kept as-is (`nn.Conv2d`'s default is also `bias=True`) for a literal
  port rather than "fixing" something the original didn't.
- **Softmax + `categorical_crossentropy` -> raw logits + `CrossEntropyLoss`:**
  mathematically equivalent, but computing softmax and cross-entropy in one fused,
  log-space operation is numerically more stable than a separate `Softmax` layer feeding
  a separate loss. Same reasoning already documented in this repo's own
  `CatDogCNN.forward` docstring.
- **`model.compile(...)`:** no PyTorch equivalent -- optimizer/loss/metrics stay as
  separate objects, used explicitly in the training loop (not written here; this
  notebook is architecture-only, see the top cell).